In [ ]:
!pip install hnswlib numpy neo4j python-dotenv openai pyvis py2neo langchain langchain_community accelerate langchain_openai tiktoken transformers asyncio nest_asyncio
!pip install --upgrade transformers

In [ ]:
import hnswlib
import numpy as np
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os
from openai import OpenAI

# Load environment variables
load_dotenv()

# Neo4j connection
DB_URI = os.getenv("DB_URI")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
HUGGINGFACE_KEY = os.getenv("HUGGINGFACE_KEY")
driver = GraphDatabase.driver(DB_URI, auth=(DB_USER, DB_PASSWORD))
# huggingface API
login(token=HUGGINGFACE_KEY,add_to_git_credential=True)
# OpenAI API key
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

# Set up LangChain components
llm = ChatOpenAI(model="gpt-4o")


In [ ]:
# Parameters
INDEX_FILE_PATH = "hnswlib_index.bin"
DIMENSION = 1536  # Dimension of the embeddings

BATCH_SIZE = 1000
CHECKPOINT_FILE = 'checkpoint_indexing.txt'
# Connect to Neo4j

In [ ]:
# Current function library
def clean_embedding(embedding_list):
    # Remove the '[' and ']' from the first and last elements respectively
    embedding_list[0] = embedding_list[0].replace('[', '')
    embedding_list[-1] = embedding_list[-1].replace(']', '')
    return np.array([float(x) for x in embedding_list])


def fetch_start_node_details(protein_name):
    query = """
MATCH (p:Protein)
WHERE p.name < $name
WITH COUNT(p) AS stepsToSkip
MATCH (specificProtein:Protein {name: $name})
RETURN specificProtein AS protein, stepsToSkip

    """
    with driver.session() as session:
        result = session.run(query, name=protein_name)
        for record in result:
            protein_details = record['protein']
            id = record['stepsToSkip']

            cleaned_embedding = clean_embedding(protein_details['embedding'])
    return protein_details, cleaned_embedding, id

def fetch_interacting_proteins(driver, protein_name):
    query = """
    MATCH (p:Protein {name: $name})
    MATCH (interactor:Protein)-[r:INTERACTS_FULL|INTERACTS_PHY]->(p)
    RETURN interactor.id AS id, interactor.name AS name, interactor.embedding AS embedding,
           interactor.annotation AS annotation, TYPE(r) AS relationshipType, r AS properties
    """
    with driver.session() as session:
        result = session.run(query, name=protein_name)
        return [{
                 "name": record["name"],
                 "embedding": clean_embedding(record["embedding"]),
                 "score": record["properties"].get('score', 'No score available'),
                 "relationshipType": record["relationshipType"]}
                for record in result]



def embedding_search_on_filtered(query_embedding, interacting_proteins, k, n_fold):
    embeddings = [protein["embedding"] for protein in interacting_proteins]

    # Calculate the total number of proteins to retrieve
    total_k = k * n_fold

    # Ensure we don't try to retrieve more proteins than available
    total_k = min(total_k, len(interacting_proteins))

    index = hnswlib.Index(space='l2', dim=DIMENSION)
    index.init_index(max_elements=len(embeddings), ef_construction=200, M=16)
    index.add_items(embeddings, list(range(len(embeddings))))

    # Retrieve total_k nearest neighbors
    labels, distances = index.knn_query(query_embedding, k=total_k)

    # Reshape labels and distances to 1D arrays
    labels = labels.flatten()
    distances = distances.flatten()

    # Create a list of (protein, distance) tuples
    protein_distance_pairs = [(interacting_proteins[label], distance) for label, distance in zip(labels, distances)]

    # Sort the pairs by distance
    protein_distance_pairs.sort(key=lambda x: x[1])

    # Separate the sorted proteins and distances
    sorted_proteins = [pair[0] for pair in protein_distance_pairs]
    sorted_distances = [pair[1] for pair in protein_distance_pairs]

    return sorted_proteins, sorted_distances

def get_proteins_for_fold(sorted_proteins, sorted_distances, k, n_fold):
    start_index = (n_fold - 1) * k
    end_index = min(n_fold * k, len(sorted_proteins))
    return sorted_proteins[start_index:end_index], sorted_distances[start_index:end_index]

def fetch_relationships_between_nodes(driver, start_node, recommended_nodes):
    recommended_node_names = [node['name'] for node in recommended_nodes]
    print("Recommended Node Names:", recommended_node_names)
    print("Start Node ID:", start_node['name'])
    query = """
    MATCH (start:Protein {name: $startName})
    MATCH (recommended:Protein) WHERE recommended.name IN $recommendedNodeNames
    OPTIONAL MATCH (start)-[r:INTERACTS_FULL | INTERACTS_PHY]->(recommended)
    RETURN start, recommended, TYPE(r) AS relationshipType, r AS properties
    """
    with driver.session() as session:
        result = session.run(query, startName=start_node['name'], recommendedNodeNames=recommended_node_names)
        relationships = []
        for record in result:
            start_node_name = record['start']['name']
            recommended_node_name = record['recommended']['name']
            recommended_node_annotation = record['recommended'].get('annotation', 'No annotation available')

            if record['properties']:  # If there are properties, it means there is a relationship
                relationships.append({
                    'start': start_node_name,
                    'end': recommended_node_name,
                    'type': record['relationshipType'],
                    'properties': {
                        'score': record['properties'].get('score', 'No score available'),
                        'annotation': recommended_node_annotation

                    }
                })
            else:
                print(f"No relationships found between {start_node_name} and {recommended_node_name}.")
    return relationships


In [ ]:
import hashlib



class GraphCache:
    def __init__(self):
        self.cache = {}
        self.precalculated_nodes = {}

    def get_cache_key(self, start_protein, k, depth, nodes_to_remove, relationship_types_to_remove, n_fold):
        key_string = f"{start_protein}_{k}_{depth}_{sorted(nodes_to_remove)}_{sorted(relationship_types_to_remove)}_{n_fold}"
        return hashlib.md5(key_string.encode()).hexdigest()

    def get(self, start_protein, k, depth, nodes_to_remove, relationship_types_to_remove, n_fold):
        key = self.get_cache_key(start_protein, k, depth, nodes_to_remove, relationship_types_to_remove, n_fold)
        return self.cache.get(key)

    def set(self, start_protein, k, depth, nodes_to_remove, relationship_types_to_remove, n_fold, proteins, relationships):
        key = self.get_cache_key(start_protein, k, depth, nodes_to_remove, relationship_types_to_remove, n_fold)
        self.cache[key] = (proteins, relationships)

    def get_precalculated_nodes(self, protein_name, k, n_fold):
        key = f"{protein_name}_{k}_{n_fold}"
        return self.precalculated_nodes.get(key)

    def set_precalculated_nodes(self, protein_name, k, n_fold, sorted_proteins, sorted_distances):
        key = f"{protein_name}_{k}_{n_fold}"
        self.precalculated_nodes[key] = (sorted_proteins, sorted_distances)

    def clear_cache(self):
        """Clear all cached graph data and precalculated nodes."""
        self.cache = {}
        self.precalculated_nodes = {}
        print("Cache and precalculated nodes cleared.")

    def get_cache_size(self):
        """Return the number of items in the cache and precalculated nodes."""
        return len(self.cache), len(self.precalculated_nodes)

graph_cache = GraphCache()


def fetch_protein_details(driver, protein_names):
    print("Fetching details for proteins:", protein_names)
    query = """
    MATCH (p:Protein)
    WHERE p.name IN $proteinNames
    RETURN p.name AS name, p.annotation AS annotation
    """
    with driver.session() as session:
        result = session.run(query, proteinNames=protein_names)
        protein_details = {}
        for record in result:
            protein_details[record['name']] = {
                'name': record['name'],
                'annotation': record.get('annotation', 'No annotation available')
            }

        # Check if any proteins were not found
        for name in protein_names:
            if name not in protein_details:
                print(f"No details found for protein: {name}")
                protein_details[name] = {
                    'name': name,
                    'annotation': 'No annotation available'
                }

    return protein_details

def prune_network(proteins, relationships, nodes_to_remove=None, relationship_types_to_remove=None):
    if nodes_to_remove:
        proteins = [p for p in proteins if p['name'] not in nodes_to_remove]
        relationships = [r for r in relationships if r['start'] not in nodes_to_remove and r['end'] not in nodes_to_remove]

    if relationship_types_to_remove:
        relationships = [r for r in relationships if r['type'] not in relationship_types_to_remove]

    return proteins, relationships


import math

def get_recommended_proteins(driver, protein_name, k=10, n_fold=1):
    precalculated = graph_cache.get_precalculated_nodes(protein_name, k, n_fold)

    if precalculated:
        sorted_proteins, sorted_distances = precalculated
    else:
        interacting_proteins = fetch_interacting_proteins(driver, protein_name)
        if not interacting_proteins:
            print(f"No interacting proteins found for {protein_name}")
            return []

        start_node_details, query_embedding, _ = fetch_start_node_details(protein_name)

        # Perform embedding search on all interacting proteins
        sorted_proteins, sorted_distances = embedding_search_on_filtered(query_embedding, interacting_proteins, k, n_fold)

        # Cache the precalculated nodes
        graph_cache.set_precalculated_nodes(protein_name, k, n_fold, sorted_proteins, sorted_distances)

    # Get the proteins for the specified fold
    recommended_proteins, _ = get_proteins_for_fold(sorted_proteins, sorted_distances, k, n_fold)

    return recommended_proteins

def expand_network(driver, start_protein, k=10, depth=2, n_fold=1):
    # Fetch details for the start protein
    start_protein_details = fetch_protein_details(driver, [start_protein])
    all_proteins = {start_protein: {**start_protein_details[start_protein], 'depth': 0}}
    all_relationships = []
    proteins_to_process = [(start_protein, 0)]

    while proteins_to_process:
        current_protein, current_depth = proteins_to_process.pop(0)

        if current_depth >= depth:
            continue

        # Determine the number of proteins to recommend based on the current depth
        num_recommendations = k if current_depth == 0 else 4
        current_n_fold = n_fold if current_depth == 0 else 1

        recommended_proteins = get_recommended_proteins(driver, current_protein, num_recommendations, current_n_fold)

        if not recommended_proteins:
            print(f"No recommended proteins found for {current_protein} at depth {current_depth}")
            continue

        start_node = {'name': current_protein}
        relationships = fetch_relationships_between_nodes(driver, start_node, recommended_proteins)
        all_relationships.extend(relationships)

        # Fetch details for recommended proteins
        recommended_protein_names = [protein['name'] for protein in recommended_proteins]
        protein_details = fetch_protein_details(driver, recommended_protein_names)

        for protein_name, details in protein_details.items():
            if protein_name not in all_proteins or all_proteins[protein_name]['depth'] > current_depth + 1:
                all_proteins[protein_name] = {
                    **details,
                    'depth': current_depth + 1
                }
                proteins_to_process.append((protein_name, current_depth + 1))

    return list(all_proteins.values()), all_relationships

def get_network(driver, start_protein, k=10, depth=2, nodes_to_remove=None, relationship_types_to_remove=None, n_fold=1):
    nodes_to_remove = nodes_to_remove or []
    relationship_types_to_remove = relationship_types_to_remove or []

    cached_result = graph_cache.get(start_protein, k, depth, nodes_to_remove, relationship_types_to_remove, n_fold)
    if cached_result:
        return cached_result

    # Expand network
    proteins, relationships = expand_network(driver, start_protein, k, depth, n_fold)

    # Prune network
    pruned_proteins, pruned_relationships = prune_network(proteins, relationships, nodes_to_remove, relationship_types_to_remove)

    # Cache the pruned result
    graph_cache.set(start_protein, k, depth, nodes_to_remove, relationship_types_to_remove, n_fold, pruned_proteins, pruned_relationships)

    return pruned_proteins, pruned_relationships

In [ ]:
from py2neo import Graph
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Connection to the Neo4j database
graph = Graph(DB_URI, auth=(DB_USER, DB_PASSWORD))

def get_color_map(num_depths):
    """Generate a color map for different depths."""
    return plt.cm.Blues(np.linspace(0.3, 1, num_depths))

def fetch_graph_data(proteins, relationships):
    nx_graph = nx.MultiDiGraph()  # Directed graph

    interaction_colors = {
        'INTERACTS_FULL': 'orange',
        'INTERACTS_PHY': 'green'
    }

    # Create a dictionary to store the minimum depth for each protein
    protein_depths = {}
    for protein in proteins:
        name = protein['name']
        depth = protein.get('depth', 0)
        if name not in protein_depths or depth < protein_depths[name]:
            protein_depths[name] = depth

    # Add nodes
    for protein in proteins:
        name = protein['name']
        depth = protein_depths[name]
        nx_graph.add_node(name, depth=depth)

    # Add edges
    for rel in relationships:
        start_name = rel['start']
        end_name = rel['end']
        interaction_type = rel['type']
        score = rel['properties'].get('score', 0)
        annotation = rel['properties'].get('annotation', 'No annotation available')

        edge_color = interaction_colors.get(interaction_type, 'gray')
        nx_graph.add_edge(start_name, end_name, weight=score/100, label=interaction_type, color=edge_color, annotation=annotation)

    return nx_graph

def visualize_network(proteins, relationships, start_protein):
    nx_graph = fetch_graph_data(proteins, relationships)

    # Determine the number of unique depths
    depths = sorted(set(nx_graph.nodes[node]['depth'] for node in nx_graph.nodes))
    num_depths = len(depths)
    color_map = get_color_map(num_depths)

    # Assign colors based on depth
    node_colors = []
    for node in nx_graph.nodes:
        if node == start_protein:
            node_colors.append('red')
        else:
            depth = nx_graph.nodes[node]['depth']
            color_index = depths.index(depth)
            node_colors.append(color_map[color_index])

    # Drawing the graph
    pos = nx.spring_layout(nx_graph, seed=42)  # Fixed seed for consistent layout
    edge_colors = [nx_graph.edges[edge]['color'] for edge in nx_graph.edges]

    plt.figure(figsize=(12, 8))
    nx.draw(nx_graph, pos, node_color=node_colors, edge_color=edge_colors, with_labels=True, node_size=700)

    # Add edge labels
    edge_labels = nx.get_edge_attributes(nx_graph, 'label')
    nx.draw_networkx_edge_labels(nx_graph, pos, edge_labels=edge_labels)

    # Add a color bar to show depth
    sm = plt.cm.ScalarMappable(cmap=plt.cm.Blues, norm=plt.Normalize(vmin=min(depths), vmax=max(depths)))
    sm.set_array([])
    cbar = plt.colorbar(sm)
    cbar.set_label('Depth')

    plt.title(f"Protein Interaction Network (Starting from {start_protein})")
    plt.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
import pickle
import os
from py2neo import Graph
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

class GraphManager:
    def __init__(self):
        self.graphs = {}
        self.current_graph = None

    def add_graph(self, name, proteins, relationships, start_protein, n_fold):
        self.graphs[name] = {
            'proteins': proteins,
            'relationships': relationships,
            'start_protein': start_protein,
            'n_fold': n_fold
        }
        self.current_graph = name
        print(f"Graph '{name}' added and set as current graph.")

    def get_graph(self, name):
        return self.graphs.get(name)

    def set_current_graph(self, name):
        if name in self.graphs:
            self.current_graph = name
            print(f"Current graph set to '{name}'.")
        else:
            print(f"Graph '{name}' not found.")
            print("Available graphs:")
            for graph_name in self.graphs.keys():
                print(f"  - '{graph_name}'")

    def list_graphs(self):
        return list(self.graphs.keys())

    def export_graph(self, name, filename=None):
        if name in self.graphs:
            if filename is None:
                filename = f"{name}.pkl"
            with open(filename, 'wb') as f:
                pickle.dump(self.graphs[name], f)
            print(f"Graph '{name}' exported to {filename}")
        else:
            print(f"Graph '{name}' not found.")

    def import_graph(self, filename):
        if os.path.exists(filename):
            with open(filename, 'rb') as f:
                imported_graph = pickle.load(f)
            name = os.path.splitext(os.path.basename(filename))[0]
            self.graphs[name] = imported_graph
            self.current_graph = name
            print(f"Graph imported from {filename} and added as '{name}'")
        else:
            print(f"File {filename} not found.")


    def visualize_current_graph(self):
        if self.current_graph:
            graph_data = self.graphs[self.current_graph]
            print(f"Visualizing graph: {self.current_graph[0]} with fold {self.current_graph[1]}")
            visualize_network(graph_data['proteins'], graph_data['relationships'], graph_data['start_protein'])
        else:
            print("No current graph set.")

    def add_node(self, name, attributes=None):
        if not self.current_graph:
            print("No current graph set.")
            return

        graph_data = self.graphs[self.current_graph]
        existing_node = next((node for node in graph_data['proteins'] if node['name'] == name), None)

        if existing_node:
            print(f"Node '{name}' already exists. Updating attributes.")
            existing_node.update(attributes or {})
        else:
            new_node = {'name': name, **(attributes or {})}
            graph_data['proteins'].append(new_node)
            print(f"Node '{name}' added to the graph.")

    def remove_node(self, name):
        if not self.current_graph:
            print("No current graph set.")
            return

        graph_data = self.graphs[self.current_graph]
        graph_data['proteins'] = [node for node in graph_data['proteins'] if node['name'] != name]
        graph_data['relationships'] = [rel for rel in graph_data['relationships']
                                       if rel['start'] != name and rel['end'] != name]
        print(f"Node '{name}' and its relationships removed from the graph.")

    def add_relationship(self, start_name, end_name, rel_type, properties=None):
        if not self.current_graph:
            print("No current graph set.")
            return

        graph_data = self.graphs[self.current_graph]
        new_rel = {
            'start': start_name,
            'end': end_name,
            'type': rel_type,
            'properties': properties or {}
        }
        graph_data['relationships'].append(new_rel)
        print(f"Relationship added: {start_name} -[{rel_type}]-> {end_name}")

    def remove_relationship(self, start_name, end_name, rel_type):
        if not self.current_graph:
            print("No current graph set.")
            return

        graph_data = self.graphs[self.current_graph]
        graph_data['relationships'] = [rel for rel in graph_data['relationships']
                                       if not (rel['start'] == start_name and
                                               rel['end'] == end_name and
                                               rel['type'] == rel_type)]
        print(f"Relationship removed: {start_name} -[{rel_type}]-> {end_name}")

    def get_node(self, name):
        if not self.current_graph:
            print("No current graph set.")
            return None

        graph_data = self.graphs[self.current_graph]
        return next((node for node in graph_data['proteins'] if node['name'] == name), None)

    def get_relationships(self, start_name=None, end_name=None, rel_type=None):
        if not self.current_graph:
            print("No current graph set.")
            return []

        graph_data = self.graphs[self.current_graph]
        return [rel for rel in graph_data['relationships']
                if (start_name is None or rel['start'] == start_name) and
                   (end_name is None or rel['end'] == end_name) and
                   (rel_type is None or rel['type'] == rel_type)]

### Build your interaction graph cache (Install all the dependencies and make sure you have run all before)

In [ ]:
# Create a graph manager instance

graph_manager = GraphManager()

def run_graph_analysis(driver, start_protein, k=10, depth=2, nodes_to_remove=None, relationship_types_to_remove=None, n_fold=1):
    proteins, relationships = get_network(driver, start_protein, k, depth, nodes_to_remove, relationship_types_to_remove,n_fold=n_fold)
    graph_name = f"{start_protein}_d{depth}_k{k}_f{n_fold}"
    graph_manager.add_graph(graph_name, proteins, relationships, start_protein, n_fold)
    return graph_name, proteins, relationships

In [ ]:
# Usage
start_protein = 'MARK4'
k = 30
depth = 2
n_fold=2
nodes_to_remove = []  # Example: remove PLIN3 from the network
relationship_types_to_remove = []  # Example: remove physical interactions

In [ ]:
graph_cache.clear_cache()# Clear the cache before running the analysis

Cache and precalculated nodes cleared.


In [ ]:
# Generate and store a graph
#start_protein: intial protein to start the graph
#k:kNN parameter
#depth: Graph size parameter 
#n_fold: Window Index
graph_name,proteins, relationship = run_graph_analysis(driver, start_protein, k=k, depth=depth, n_fold=1)


In [ ]:
graph_name_,proteins_, relationship_ = run_graph_analysis(driver, start_protein, k=10, depth=2)

Graph 'MARK4_d2_k10' added and set as current graph.


In [ ]:
import json
# Create a dictionary with the results
results_dict = {
    'Graph name': graph_name,
    'Protein': proteins,
    'Relationship': relationship
}

# Export to JSON
json_path = 'Raw_graph_without_query_k30_d2_MAPT.json'
with open(json_path, 'w') as f:
    json.dump(results_dict, f)
print(f"Results exported to {json_path}")

In [ ]:
import json

# Path to your JSON file
json_path = 'Raw_graph_without_query_MARK4_d2_k20_f1.json'

# Load the JSON file
with open(json_path, 'r') as f:
    data = json.load(f)

# Assign the 'Protein' and 'Relationship' data to variables
proteins = data['Protein']
relationship = data['Relationship']

In [ ]:
graph_manager = GraphManager()
graph_manager.add_graph('k30_d2_MAPT', proteins, relationship, 'MAPT',n_fold=1)

Graph 'k30_d2_MAPT' added and set as current graph.


In [ ]:
graph_manager.list_graphs()

In [ ]:
# Visualize the current graph
graph_manager.set_current_graph('MARK4_d2_k30')
graph_manager.visualize_current_graph()

## Impact search

In [ ]:

# After you load the graph or generate your graphs, you can interact with it using the following methods:

import json
import networkx as nx
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline,AutoConfig
import torch
from huggingface_hub import login
import asyncio
from concurrent.futures import ThreadPoolExecutor
import sys
import numpy as np
import time
import random
import re
import nest_asyncio
from typing import Callable, Any
from accelerate import Accelerator

# huggingface API
login(token=HUGGINGFACE_KEY,add_to_git_credential=True)

## Data Loading and Graph Construction

def load_protein_data(protein_data: str) -> Dict[str, Dict]:
    proteins = {}
    for line in protein_data.split('\n'):
        if line.strip():
            protein = json.loads(line)
            proteins[protein['name']] = protein
    return proteins

def load_relationship_data(relationship_data: str) -> List[Dict]:
    relationships = []
    for line in relationship_data.split('\n'):
        if line.strip():
            relationship = json.loads(line)
            relationships.append(relationship)
    return relationships

def build_graph(proteins: List[Dict], relationships: List[Dict]) -> nx.Graph:
    G = nx.Graph()
    for protein in proteins:
        G.add_node(protein['name'], **protein)

    for relationship in relationships:
        start, end = relationship['start'], relationship['end']
        G.add_edge(start, end, **relationship)
    return G





## Path Finding and Scoring
# Function to ensure prints are immediately visible
def print_flush(*args, **kwargs):
    print(*args, **kwargs)
    sys.stdout.flush()


def extract_string_value(value: Any) -> str:
    if hasattr(value, 'text'):
        return value.text
    elif hasattr(value, 'content'):
        return str(value.content)
    elif isinstance(value, str):
        return value
    else:
        return str(value)


def clean_json_output(raw_output: str) -> dict:
    # First, try to parse the input as is
    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        pass

    # If that fails, try to clean it as JSON
    cleaned_output = re.sub(r'^```json|```$', '', raw_output).strip()
    try:
        return json.loads(cleaned_output)
    except json.JSONDecodeError:
        pass

    # If JSON parsing fails, try to evaluate it as a Python literal
    try:
        return ast.literal_eval(cleaned_output)
    except (ValueError, SyntaxError):
        pass

    # If all parsing attempts fail, raise an error
    raise ValueError(f"Unable to parse output as JSON or dict: {raw_output}")



async def retry_with_exponential_backoff(
    func: Callable,
    *args,
    max_retries: int = 50,
    base_delay: float = 1,
    rate_limit_errors: tuple = ('rate_limit_exceeded', 'too_many_requests')
):
    retries = 0
    while True:
        try:
            return await func(*args)
        except Exception as e:
            error_message = str(e).lower()
            is_rate_limit_error = any(err in error_message for err in rate_limit_errors)

            if not is_rate_limit_error or retries >= max_retries:
                print(f"Error occurred: {e}")
                raise

            # delay = base_delay * (2 ** retries) + random.uniform(0, 1)
            delay = base_delay
            print(f"Rate limit exceeded. Retrying in {delay:.2f} seconds...")
            await asyncio.sleep(delay)
            retries += 1

def find_paths(G: nx.Graph, start: str, max_length: int = 4) -> List[List[str]]:
    paths = []
    for node in G.nodes():
        paths.extend(nx.all_simple_paths(G, start, node, cutoff=max_length))
    return paths


async def extract_json_from_string(s):
    try:
        # Find the start and end of the JSON object
        start = s.find('{')
        end = s.rfind('}') + 1
        if start != -1 and end != -1:
            json_str = s[start:end]
            return json.loads(json_str)
    except json.JSONDecodeError:
        pass
    return None

async def score_edge(G: nx.Graph, start: str, end: str, query: str) -> str:
    start_annotation = G.nodes[start]['annotation']
    end_annotation = G.nodes[end]['annotation']

    try:
        print_flush(f"Scoring edge {start} -> {end}")
        result = await retry_with_exponential_backoff(
            edge_relevance_chain.ainvoke,
            {
                "query": query,
                "start_protein": start,
                "end_protein": end,
                "start_annotation": start_annotation,
                "end_annotation": end_annotation
            }
        )

        # Extract string value from result
        result = extract_string_value(result)

        print_flush(f"Debug - LLM output for edge {start} -> {end}:")
        print_flush(f"Content: {result}")

        # No need to parse JSON here, just return the explanation
        explanation = result

    except Exception as e:
        print_flush(f"Error in scoring edge {start} -> {end}: {str(e)}")
        explanation = f"Error in scoring: {str(e)}"

    return explanation

async def score_path(G: nx.Graph, path: List[str], query: str) -> Tuple[float, str, str]:
    path_details = ""

    edge_tasks = []
    for i in range(len(path) - 1):
        start, end = path[i], path[i+1]
        edge_tasks.append(score_edge(G, start, end, query))

    edge_results = await asyncio.gather(*edge_tasks)

    for i, edge_explanation in enumerate(edge_results):
        start, end = path[i], path[i+1]
        path_details += f"{start} -> {end}:\n{edge_explanation}\n\n"

    try:
        print_flush("Generating path explanation")
        path_explanation_result = await retry_with_exponential_backoff(
            path_explanation_chain.ainvoke,
            {
                "query": query,
                "path": " -> ".join(path),
                "path_details": path_details
            }
        )

        # Extract string value from result
        path_explanation_result = extract_string_value(path_explanation_result)

        parsed_result = clean_json_output(path_explanation_result)

        path_explanation = parsed_result['explanation']
        path_score = float(parsed_result.get('relevance_score', 0))

        if not 0 <= path_score <= 100:
            print_flush(f"Warning: Relevance score out of range (0-100) for path explanation: {path_score}")
            path_score = max(0, min(100, path_score))  # Clamp the score between 0 and 100

    except Exception as e:
        print_flush(f"Error in generating path explanation: {str(e)}")
        path_explanation = f"Error in generating explanation: {str(e)}"
        path_score = 0

    return path_score, path_explanation, path_details
# New Path Generation Function Using Raw Annotations

async def score_path_with_annotations(path: list, protein_annotations: list, query: str) -> tuple:
    # Build node annotations string
    node_annotations_str = ""
    for node, annotation in zip(path, protein_annotations):
        node_annotations_str += f"{node}: {annotation}\n"

    try:
        print(f"Generating path explanation using node annotations for path {' -> '.join(path)}")
        path_explanation_result = await retry_with_exponential_backoff(
            path_explanation_chain_with_annotations.ainvoke,
            {
                "query": query,
                "path": " -> ".join(path),
                "node_annotations": node_annotations_str
            }
        )

        # Extract string value from result
        path_explanation_result = extract_string_value(path_explanation_result)

        parsed_result = clean_json_output(path_explanation_result)

        path_explanation = parsed_result['explanation']
        path_score = float(parsed_result.get('relevance_score', 0))

        if not 0 <= path_score <= 100:
            print(f"Warning: Relevance score out of range (0-100) for path explanation: {path_score}")
            path_score = max(0, min(100, path_score))  # Clamp the score between 0 and 100

    except Exception as e:
        print(f"Error in generating path explanation: {str(e)}")
        path_explanation = f"Error in generating explanation: {str(e)}"
        path_score = 0

    return path_score, path_explanation


async def async_recommend_paths(G: nx.Graph, query: str, start_protein: str, max_length: int = 5, top_k: int = 20) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    paths = find_paths(G, start_protein, max_length)
    print_flush(f"Found {len(paths)} paths.")

    scored_paths = []
    for path in paths:
        if len(path) < 2:
            continue
        path_score, path_explanation, path_details = await score_path(G, path, query)
        scored_paths.append((path, path_score, path_explanation, path_details))

    scored_paths.sort(key=lambda x: x[1], reverse=True)
    top_paths = scored_paths[:top_k]

    recommended_proteins = []
    recommended_relationships = []
    recommended_path_details = []

    for path, path_score, path_explanation, path_details in top_paths:
        path_proteins = []
        path_relationships = []

        # Parse path_details to extract edge explanations
        edge_explanations = {}
        print_flush(f"Path details: {path_details}")
        print_flush(f"Type of path_details: {type(path_details)}")
        for detail in path_details.split('\n'):

            # if ' -> ' in detail and ':' in detail:
            #     edge_key, explanation = detail.split(':', 1)
            #     edge_explanations[edge_key.strip()] = explanation.strip()

            #     print_flush('EDGE KEY:', edge_key)
            #     print_flush('EXPLANATION:', edge_explanations)
              if ' -> ' in detail and detail.endswith(':'):
                edge_key = detail.strip()[:-1]  # Remove the colon at the end
                explanation_start = path_details.index(detail) + len(detail)
                explanation_end = path_details.find('\n\n', explanation_start)
                if explanation_end == -1:  # If it's the last explanation
                    explanation_end = len(path_details)
                explanation_text = path_details[explanation_start:explanation_end].strip()
                edge_explanations[edge_key] = (explanation_text)
                print_flush('EDGE KEY:', edge_key)
                print_flush('EXPLANATION:', edge_explanations)

        for protein in path:
            if protein not in [p['name'] for p in recommended_proteins]:
                protein_info = {
                    'name': protein,
                    'annotation': G.nodes[protein]['annotation'],
                    'depth': G.nodes[protein]['depth']
                }
                recommended_proteins.append(protein_info)
                path_proteins.append(protein_info)

        for i in range(len(path) - 1):
            start, end = path[i], path[i+1]
            edge_data = G[start][end]
            edge_key = f"{start} -> {end}"

            explanation = edge_explanations.get(edge_key, "No explanation available")

            relationship_info = {
                'start': start,
                'end': end,
                'type': edge_data['type'],
                'properties': {
                    'score': edge_data['properties']['score'],
                    'annotation': edge_data['properties']['annotation']
                },
                'explanation': explanation
            }
            recommended_relationships.append(relationship_info)
            path_relationships.append(relationship_info)

        recommended_path_details.append({
            'path': path,
            'score': path_score,
            'explanation': path_explanation,
            'details': path_details,
            'proteins': path_proteins,
            'relationships': path_relationships
        })

    return recommended_proteins, recommended_relationships, recommended_path_details
# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

def run_recommendation(G: nx.Graph, query: str, start_protein: str, max_length: int = 5, top_k: int = 20) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    async def run_async():
        task = asyncio.create_task(async_recommend_paths(G, query, start_protein, max_length, top_k))
        try:
            return await task
        except asyncio.CancelledError:
            print("Task was cancelled due to keyboard interrupt.")
            return None, None, None

    loop = asyncio.get_event_loop()
    main_task = asyncio.ensure_future(run_async())

    try:
        return loop.run_until_complete(main_task)
    except KeyboardInterrupt:
        print("Keyboard interrupt detected. Cancelling the task...")
        main_task.cancel()
        loop.run_until_complete(main_task)  # Allow task to be cancelled
        return None, None, None



## Visualization Function

def get_color_map(num_depths):
    """Generate a color map for different depths."""
    return plt.cm.Blues(np.linspace(0.3, 1, num_depths))

def visualize_network(proteins, relationships, start_protein):
    nx_graph = nx.MultiDiGraph()

    interaction_colors = {
        'INTERACTS_FULL': 'orange',
        'INTERACTS_PHY': 'green'
    }

    # Create a dictionary to store the minimum depth for each protein
    protein_depths = {}
    for protein in proteins:
        name = protein['name']
        depth = protein.get('depth', 0)
        if name not in protein_depths or depth < protein_depths[name]:
            protein_depths[name] = depth

    # Add nodes
    for protein in proteins:
        name = protein['name']
        depth = protein_depths[name]
        nx_graph.add_node(name, depth=depth)

    # Add edges
    for rel in relationships:
        start_name = rel['start']
        end_name = rel['end']
        interaction_type = rel['type']
        score = rel['properties'].get('score', 0)
        annotation = rel['properties'].get('annotation', 'No annotation available')

        edge_color = interaction_colors.get(interaction_type, 'gray')
        nx_graph.add_edge(start_name, end_name, weight=score/100, label=interaction_type, color=edge_color, annotation=annotation)

    # Determine the number of unique depths
    depths = sorted(set(nx_graph.nodes[node]['depth'] for node in nx_graph.nodes))
    num_depths = len(depths)
    color_map = get_color_map(num_depths)

    # Assign colors based on depth
    node_colors = []
    for node in nx_graph.nodes:
        if node == start_protein:
            node_colors.append('red')
        else:
            depth = nx_graph.nodes[node]['depth']
            color_index = depths.index(depth)
            node_colors.append(color_map[color_index])

    # Drawing the graph
    pos = nx.spring_layout(nx_graph, seed=42)  # Fixed seed for consistent layout
    edge_colors = [nx_graph.edges[edge]['color'] for edge in nx_graph.edges]

    plt.figure(figsize=(12, 8))
    nx.draw(nx_graph, pos, node_color=node_colors, edge_color=edge_colors, with_labels=True, node_size=700)

    # Add edge labels
    edge_labels = nx.get_edge_attributes(nx_graph, 'label')
    nx.draw_networkx_edge_labels(nx_graph, pos, edge_labels=edge_labels)

    # Add a color bar to show depth
    sm = plt.cm.ScalarMappable(cmap=plt.cm.Blues, norm=plt.Normalize(vmin=min(depths), vmax=max(depths)))
    sm.set_array([])
    cbar = plt.colorbar(sm)
    cbar.set_label('Depth')

    plt.title(f"Recommended Protein Interaction Network (Starting from {start_protein})")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
###################
llm = ChatOpenAI(model="gpt-4o")
###############


# # Check GPU availability
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# # Load model and tokenizer
# model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name)



In [ ]:

from langchain_community.llms import HuggingFacePipeline

# # Create pipeline with GPU
# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     device=0 if torch.cuda.is_available() else -1,  # Use GPU if available
#     torch_dtype=torch.float16,
#     max_length=2048
# )

# llm = HuggingFacePipeline(pipeline=pipe)

edge_relevance_prompt = PromptTemplate(
    input_variables=["query", "start_protein", "end_protein", "start_annotation", "end_annotation"],
    template="""As a protein-protein interaction expert, evaluate the relevance of the interaction between {start_protein} and {end_protein}
    to the query: "{query}".

       You should follow those steps for analysis:
        <Steps>
        1. Use the name of proteins along the path to figure out the directional graph.
        Consider the direction of the interaction and always and only consider the impact of the
        direct last protein. The effect propogates backward to the start protein.
        2. The previous protein should be able to directly interact with the current protein according to
        the information provided. If not, the interaction is not relevant.
        3. Explan potential biological processes or mechanisms before recommending it. If you can not
        find any explict reasons to indicate that they are relevant, the path will be irrelevant.
        4. The proteins on the ends of one edge must be able to directly interact with each other.
        If not, the edge does not make any sense and the relevance is zero.
        5. The effect propogates from the end protein to the start protein.
        </Steps>
        Protein information:
        {start_protein}: {start_annotation}
        {end_protein}: {end_annotation}
        Explain why this interaction is relevant or not relevant to the query using less than 35 words.
        Try to explain the edge informatively using simple and short sentence structure to improve the readability.



    """

)

edge_relevance_chain = (
    {"query":RunnablePassthrough(),"start_protein":RunnablePassthrough(), "end_protein":RunnablePassthrough(), "start_annotation":RunnablePassthrough(), "end_annotation":RunnablePassthrough()}
    | edge_relevance_prompt
    | llm
    | StrOutputParser()
)

path_explanation_prompt = PromptTemplate(
    input_variables=["query", "path", "path_details"],
    template="""As a protein-protein interaction expert, choose the following protein interaction paths to satisfy
            the query: "{query}"
            You should follow those steps for analysis:
            <Steps>
            1. Use the name of proteins along the path to figure out the directional graph.
            Protein A -> Protein B -> Protein C means that protein c can influence protein b and protein b
            can influence protein a. Any other directions are not relevant.
            2. The path should satisfy the requirements in the query. The more likely the requirements get
            satisfied, the higher the relevance score is.
            3. The previous protein should be able to directly interact with the current protein according to
            the information provided. If not, the interaction is not relevant.
            4. Explain potential biological processes or mechanisms before recommending it. If you can not
            find any explict reasons to indicate that they are relevant, the path will be irrelevant.
            5. Provide a relevance score from 0 to 100, where 0 is not relevant at all and 100 is highly
            relevant. This score will be used to rank the paths according to their relevance of
            the explanation to the query.
            6. The proteins on the ends of one edge must be able to directly interact with each other. If not,
            the path does not make any sense and the relevance is zero.
            7. The effect propogates from end protein to start protein.
            </Steps>
            Path: {path}
            Details of each step: {path_details}
            Discuss any potential biological processes or mechanisms that this path might represent and
            explain why you recommend this path, using less than 80 words.
            Your explanation should be informative and accessible, as if you are explaining it to a
            fellow researcher. Your response MUST be in the following JSON format:
            {{
            "explanation": "Your explanation here",
            "relevance_score": "0-100"
                }}

    """

)


path_explanation_chain = (
    {"query":RunnablePassthrough(),"path":RunnablePassthrough(), "path_details":RunnablePassthrough()}
    | path_explanation_prompt
    | llm
    | StrOutputParser()
)


In [ ]:
import json

# Path to your Interaction Graph JSON file
json_path = 'Raw_graph_without_query_MARK4_d2_k20_f1.json'
# Load the JSON file
with open(json_path, 'r') as f:
    data = json.load(f)

# Assign the 'Protein' and 'Relationship' data to variables
proteins = data['Protein']
relationship = data['Relationship']

proteins = proteins
relationships = relationship
G = build_graph(proteins, relationships)

In [ ]:
# Usage in Jupyter notebook:
start_protein = 'MARK4'
query = "Find protein paths that the end protein phosphorylates MARK4"
results = run_recommendation(G, query, start_protein)

if results[0] is not None:
    recommended_proteins, recommended_relationships, recommended_path_details = results
    # Process the results here
else:
    print("No results obtained due to interruption.")




In [ ]:
# Create a dictionary with the results
results_dict = {
    'Recommended Proteins': recommended_proteins,
    'Relationships': recommended_relationships,
    'Path Details': recommended_path_details
}

# Export to JSON
json_path = 'MARK4.json'
with open(json_path, 'w') as f:
    json.dump(results_dict, f)
print(f"Results exported to {json_path}")

Results exported to NewMARK4.json


In [ ]:
import json

# Path to your JSON file
json_path = 'MARK4.json'

# Load the JSON file
with open(json_path, 'r') as f:
    results = json.load(f)

# Assign the 'Protein' and 'Relationship' data to variables
proteins = results['Recommended Proteins']
relationship = results['Relationships']
recommended_path_details = results['Path Details']

In [ ]:
print(f"\nTop recommendations for query: '{query}'")

print("\nRecommended Proteins:")
for protein in recommended_proteins:
    print(f"Name: {protein['name']}")
    print(f"Annotation: {protein['annotation']}")
    print(f"Depth: {protein['depth']}")
    print()




In [ ]:
print("\nRecommended Relationships:")
for rel in recommended_relationships:
    print(f"Start: {rel['start']}")
    print(f"End: {rel['end']}")
    print(f"Type: {rel['type']}")
    print(f"Score: {rel['properties']['score']}")
    print(f"Annotation: {rel['properties']['annotation']}")
    print(f"Explanation: {rel['explanation']}")
    print()



In [ ]:
print("\nRecommended Path Details:")
for path_detail in recommended_path_details:
    print(f"Path: {' -> '.join(path_detail['path'])}")
    print(f"Score: {path_detail['score']}")
    print(f"Explanation: {path_detail['explanation']}")
    print("Detailed protein information:")
    for protein in path_detail['proteins']:
        print(f"  {protein['name']}: {protein['annotation']} (Depth: {protein['depth']})")
    print("Detailed relationship information:")
    for rel in path_detail['relationships']:
        print(f"  {rel['start']} -> {rel['end']} ({rel['type']}):")
        print(f"    Explanation: {rel['explanation']}")
    print()


In [ ]:
visualize_network(recommended_proteins, recommended_relationships, start_protein)